In [ ]:
import torch
from torch.distributions import Categorical

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# build all combos
dims = [3,4,5,6]
mask = [0,1,0,1]
keep_blocks = torch.tensor(mask, dtype=torch.bool)

B = 4
logits = torch.randn(B, sum(dims), device=device)

grids = torch.cartesian_prod(*[torch.arange(d, device=device) for d in dims])  # [315, 4]
# keep only combos with masked blocks == 0
mask_rows = torch.ones(len(dims), dtype=torch.bool, device=device)
mask_rows = ~keep_blocks
valid = (grids[:, mask_rows] == 0).all(dim=-1)
grids = grids[valid]  # [315', 4], here 315' = 3*5*1*1 = 15

# compute joint logits for each combo by summing per-block logits
offsets = torch.nn.functional.pad(torch.tensor(dims, device=device).cumsum(0), (1,0))[:-1]
pieces = []
for i, d in enumerate(dims):
    start = offsets[i]
    block_logits = logits[:, start:start+d]                      # [B, d]
    idx = grids[:, i].unsqueeze(0).expand(logits.size(0), -1)    # [B, 315']
    pieces.append(block_logits.gather(1, idx))                   # [B, 315']
joint_logits = sum(pieces)                                       # [B, 315']

joint_dist = Categorical(logits=joint_logits)                    # batched
joint_sample = joint_dist.sample()                               # [B], each in [0..315'-1]
# convert back to per-block indices:
idxs = grids[joint_sample]                                       # [B, 4], columns 2,3 are 0

print(idxs)

In [ ]:
import numpy as np

print(np.cartesian_prod([np.arange(3), np.arange(4)]))

In [1]:
import numpy as np

def cartesian_prod(arrays, *, dtype=None):
    """
    arrays: list/tuple of 1D numpy arrays
    returns: (prod(len(a) for a in arrays), len(arrays)) ndarray
    """
    arrays = [np.asarray(a).ravel() for a in arrays]
    if not arrays:
        raise ValueError("arrays must be a non-empty list of 1D arrays")

    # If any input is empty, the product is empty with the right number of columns
    if any(a.size == 0 for a in arrays):
        n = len(arrays)
        dt = dtype if dtype is not None else np.result_type(*arrays)
        return np.empty((0, n), dtype=dt)

    grids = np.meshgrid(*arrays, indexing='ij')
    out = np.stack(grids, axis=-1).reshape(-1, len(arrays))
    if dtype is not None:
        out = out.astype(dtype, copy=False)
    return out


# Create a grid of indices for the discrete parameters
grid = cartesian_prod([np.arange(3), np.arange(4), np.arange(5)])

# Dimension Mask
dimension_mask = [0,1,1]

print(grid)
print(dimension_mask)

[[0 0 0]
 [0 0 1]
 [0 0 2]
 [0 0 3]
 [0 0 4]
 [0 1 0]
 [0 1 1]
 [0 1 2]
 [0 1 3]
 [0 1 4]
 [0 2 0]
 [0 2 1]
 [0 2 2]
 [0 2 3]
 [0 2 4]
 [0 3 0]
 [0 3 1]
 [0 3 2]
 [0 3 3]
 [0 3 4]
 [1 0 0]
 [1 0 1]
 [1 0 2]
 [1 0 3]
 [1 0 4]
 [1 1 0]
 [1 1 1]
 [1 1 2]
 [1 1 3]
 [1 1 4]
 [1 2 0]
 [1 2 1]
 [1 2 2]
 [1 2 3]
 [1 2 4]
 [1 3 0]
 [1 3 1]
 [1 3 2]
 [1 3 3]
 [1 3 4]
 [2 0 0]
 [2 0 1]
 [2 0 2]
 [2 0 3]
 [2 0 4]
 [2 1 0]
 [2 1 1]
 [2 1 2]
 [2 1 3]
 [2 1 4]
 [2 2 0]
 [2 2 1]
 [2 2 2]
 [2 2 3]
 [2 2 4]
 [2 3 0]
 [2 3 1]
 [2 3 2]
 [2 3 3]
 [2 3 4]]
[0, 1, 1]


In [3]:
def get_logit_mask(reaction_len, dimensions, parameter_mask):
        if parameter_mask is None:
            return None
        grid = cartesian_prod([np.arange(d) for d in dimensions]) 
        logit_mask = np.ones((reaction_len, grid.shape[0]), dtype=bool)
        for j in range(reaction_len):
            for i in range(len(parameter_mask[j])):
                if parameter_mask[j, i] == 0:
                    logit_mask[j] = logit_mask[j] & (grid[:,i] == 0)
        return logit_mask

get_logit_mask(3, [3,4,5], np.array([[0,1,0], [1,1,1], [0,0,1]]))

array([[ True, False, False, False, False,  True, False, False, False,
        False,  True, False, False, False, False,  True, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False],
       [ True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True, Fal

In [4]:
print(logit_mask.shape)
print(type(grid))

(60,)
<class 'numpy.ndarray'>
